# 项目一：数据理解与任务定义

任务：用学期早期可获得的信息识别最终可能不及格的学生。

本项目把最终成绩 `G3 < 10` 定义为风险学生。重点不是追求漂亮分数，而是建立一个没有信息泄漏、能用于早期干预的流程。

In [1]:
from pathlib import Path

import pandas as pd

# 网页 JupyterLab 从项目根目录启动；批量执行时内核可能从用户目录启动，因此保留本机回退路径。
data_path = Path('projects/01_student_risk_prediction/data/raw/student-mat.csv')
if not data_path.exists():
    data_path = Path.home() / 'solo_work/算法工程师/projects/01_student_risk_prediction/data/raw/student-mat.csv'
df = pd.read_csv(data_path, sep=';')

print(f'数据形状：{df.shape}')
display(df.head())

数据形状：(395, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [2]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'unique_count': df.nunique(),
})
display(summary)

print(f'重复行数量：{df.duplicated().sum()}')

,dtype,missing_count,unique_count
school,object,0,2
sex,object,0,2
age,int64,0,8
address,object,0,2
famsize,object,0,2
Pstatus,object,0,2
Medu,int64,0,5
Fedu,int64,0,5
Mjob,object,0,5
Fjob,object,0,5


重复行数量：0


## 标签与特征边界

- `G3`：最终成绩，用来生成风险标签。
- `G1`、`G2`：前两阶段成绩。它们在预测时点不可用，因此必须排除。
- 其他字段：可作为早期风险判断的候选特征，但仍要结合现实含义审查。

In [3]:
# 1 表示最终不及格，作为高风险正类。
y = (df['G3'] < 10).astype(int)

# 防止目标泄漏：G3 构造标签，G1/G2 也不属于早期可用信息。
leakage_columns = ['G1', 'G2', 'G3']
X = df.drop(columns=leakage_columns)

# value_counts() 统计每个标签出现的人数；sort_index() 确保按 0、1 顺序展示。
# 这里 0 表示最终及格，1 表示最终不及格（高风险）。
risk_summary = pd.DataFrame({
    'count': y.value_counts().sort_index(),
    # normalize=True 会把人数除以总人数，得到每类所占比例。
    'ratio': y.value_counts(normalize=True).sort_index(),
    # 指定完整索引 0、1，并给索引起名 at_risk，保证表格语义清晰。
}, index=pd.Index([0, 1], name='at_risk'))
display(risk_summary)
# X.shape 是（样本数，特征数）；索引 1 取第二个数，即特征列数量。
print(f'用于建模的特征数量：{X.shape[1]}')

,count,ratio
at_risk,,
0,265,0.670886
1,130,0.329114


用于建模的特征数量：30


In [4]:
numeric_features = X.select_dtypes(include='number').columns.tolist()
categorical_features = X.select_dtypes(exclude='number').columns.tolist()

print(f'数值特征（{len(numeric_features)} 个）：{numeric_features}')
print(f'类别特征（{len(categorical_features)} 个）：{categorical_features}')

数值特征（13 个）：['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']
类别特征（17 个）：['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


## 本阶段结论

下一步会只在训练集上学习数值填补、标准化与类别编码规则，再将同一规则应用到验证集和测试集。由于风险学生比例通常较低，后续不能只看 Accuracy，还要重点关注 Recall、Precision、F1、ROC-AUC。

思考题：为什么在这个项目中，即使 `G1` 和 `G2` 能大幅提高分数，也仍要把它们从早期风险模型的特征中删除？